# Modelos de recomendação e calibração

## Configurações iniciais

In [1]:
import pandas as pd
import spotipy
from time import sleep
import json
import os

from pyspark.sql import SparkSession
os.environ['SPARK_HOME'] = '/home/david/Documentos/UFABC/PGC/Codigos/code/Spark'
os.environ['PYSPARK_DRIVER_PYTHON'] = 'jupyter' 
os.environ['PYSPARK_DRIVER_OPTS'] = 'notebook'
os.environ['PYSPARK_PYTHON'] = 'python'

In [2]:
spark = SparkSession.builder \
        .appName('PGC') \
        .getOrCreate()

24/08/08 11:50:46 WARN Utils: Your hostname, david-Nitro resolves to a loopback address: 127.0.1.1; using 192.168.15.9 instead (on interface wlp9s0)
24/08/08 11:50:46 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/08/08 11:50:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Modelos de recomendação

### Filtragem colaborativa (ALS)

- [Recommender System using ALS in Pyspark](https://medium.com/@brunoborges_38708/recommender-system-using-als-in-pyspark-10329e1d1ee1)
- [Sistema de recomendação de filmes com filtragem colaborativa usando Spark e ALS](https://medium.com/camilawaltrick/sistema-de-recomendacao-filtragem-colaborativa-als-spark-f5a4a7ccf8cf)
    * [Github do artigo citado a cima](https://github.com/cwaltrick/data_science/blob/master/Modelo_ALS.ipynb)


#### Importações necessárias

In [3]:
from pyspark.ml.evaluation import RegressionEvaluator #evaluation é a biblioteca para verificação da qualidade do modelo
from pyspark.ml.recommendation import ALS # ALS é o modelo de recomendação que será utilizadp
from pyspark.sql import Row #row é o formato que o ALS trabalha, row conterá o id do usuario, id filme, nota e timestamp

#### Carregando dataset do MovieLens

In [4]:
df = pd.read_csv('../data/raw/movielens/100K/ml-100k/u.data',
                names=['userId', 'movieId', 'rating', 'timestamp'],
                sep='\t')

In [5]:
df = spark.read.csv("../data/raw/movielens/100K/ml-100k/u.data", inferSchema = True, sep='\t')
header = ['userId', 'movieId', 'rating', 'timestamp']
df = df.toDF(*header)

#### Criando o modelo

Separação simples entre treino e teste (80% treino e 20% teste)

In [6]:
(training, test) = df.randomSplit([0.8, 0.2])

Modelo ALS

Parâmetros: quantidade máxima de iterações, coeficiente de aprendizado, as colunas utilizadas e desconsidera o usuário que tiver coldstart, caso ocorra.

In [7]:
als = ALS(maxIter=5, regParam=0.01, userCol="userId", itemCol="movieId", ratingCol="rating", coldStartStrategy="drop")

Treinando o modelo com a porção de treino

In [8]:
model = als.fit(training)

24/08/08 11:50:57 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS
24/08/08 11:51:05 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


Realizando testes

In [38]:
predictions = model.transform(test)

In [39]:
evaluator = RegressionEvaluator(metricName="rmse", labelCol="rating",
predictionCol="prediction")
rmse = evaluator.evaluate(predictions)
print("Erro médio quadrático = " + str(rmse))

Erro médio quadrático = 1.0687315206644807


Pegar todos os usuários e gerar 10 recomendações

In [9]:
userRec = model.recommendForAllUsers(10) 

In [10]:
userRec.show()

+------+--------------------+
|userId|     recommendations|
+------+--------------------+
|     1|[{361, 6.4711213}...|
|     3|[{565, 8.496407},...|
|     5|[{853, 7.382985},...|
|     6|[{624, 5.6898465}...|
|     9|[{1483, 9.217169}...|
|    12|[{1160, 8.612247}...|
|    13|[{1286, 6.1991076...|
|    15|[{337, 8.555535},...|
|    16|[{1262, 6.6797495...|
|    17|[{695, 9.088854},...|
|    19|[{320, 7.1868815}...|
|    20|[{574, 10.646606}...|
|    22|[{548, 8.392521},...|
|    26|[{361, 4.9222593}...|
|    27|[{1286, 9.770725}...|
|    28|[{320, 5.8429494}...|
|    31|[{361, 7.963117},...|
|    34|[{1069, 10.045025...|
|    35|[{1273, 8.505535}...|
|    37|[{904, 10.746496}...|
+------+--------------------+
only showing top 20 rows



In [14]:
userRecsOnlyItemId = userRec.select(userRec['userId'], userRec['recommendations']['movieid'])     

In [15]:
userRecsOnlyItemId.show(10, False)

+------+--------------------------------------------------------+
|userId|recommendations.movieid                                 |
+------+--------------------------------------------------------+
|1     |[361, 1159, 1150, 610, 962, 1449, 359, 906, 958, 114]   |
|3     |[565, 1271, 426, 944, 1419, 548, 850, 571, 1470, 800]   |
|5     |[853, 361, 464, 1199, 130, 610, 1163, 1207, 800, 919]   |
|6     |[624, 1131, 850, 548, 361, 1313, 1286, 1643, 1449, 1463]|
|9     |[1483, 787, 1631, 904, 518, 915, 464, 962, 1103, 1245]  |
|12    |[1160, 267, 793, 1273, 371, 1167, 1192, 394, 1483, 888] |
|13    |[1286, 667, 592, 718, 1463, 1172, 956, 961, 1153, 502]  |
|15    |[337, 1503, 835, 1022, 1155, 641, 1463, 726, 694, 555]  |
|16    |[1262, 954, 842, 836, 1311, 1169, 634, 1643, 1070, 1103]|
|17    |[695, 1203, 464, 1380, 1240, 536, 1262, 1553, 904, 764] |
+------+--------------------------------------------------------+
only showing top 10 rows



In [17]:
rec_df_pd = userRecsOnlyItemId.toPandas()

In [21]:
rec_df_pd.set_index('userId')['recommendations.movieid'].to_dict()

{1: [361, 1159, 1150, 610, 962, 1449, 359, 906, 958, 114],
 3: [565, 1271, 426, 944, 1419, 548, 850, 571, 1470, 800],
 5: [853, 361, 464, 1199, 130, 610, 1163, 1207, 800, 919],
 6: [624, 1131, 850, 548, 361, 1313, 1286, 1643, 1449, 1463],
 9: [1483, 787, 1631, 904, 518, 915, 464, 962, 1103, 1245],
 12: [1160, 267, 793, 1273, 371, 1167, 1192, 394, 1483, 888],
 13: [1286, 667, 592, 718, 1463, 1172, 956, 961, 1153, 502],
 15: [337, 1503, 835, 1022, 1155, 641, 1463, 726, 694, 555],
 16: [1262, 954, 842, 836, 1311, 1169, 634, 1643, 1070, 1103],
 17: [695, 1203, 464, 1380, 1240, 536, 1262, 1553, 904, 764],
 19: [320, 361, 962, 1131, 1643, 113, 800, 375, 1181, 267],
 20: [574, 786, 1185, 902, 745, 767, 1311, 374, 1540, 668],
 22: [548, 361, 800, 904, 1643, 1207, 610, 1199, 906, 478],
 26: [361, 1643, 1463, 963, 1131, 1642, 1449, 1367, 945, 251],
 27: [1286, 667, 464, 592, 1160, 1218, 1273, 1113, 502, 400],
 28: [320, 853, 361, 1643, 1004, 1169, 836, 1065, 253, 1019],
 31: [361, 320, 1181, 962

### Filtragem baseada em conteúdo (k-NN)

[Github para se inspirar](https://github.com/jisilvia/kNN_Recommender_System/blob/main/kNN_Recommender_System.ipynb)

## Calibração

### Obtenção da distribuição de gêneros (perfil do usuário)

Nessa etapa, vamos criar um método para visualizar a quantidade de itens que o usuário interagiu para cada gênero de itens.

Para facilitar o uso da informação de gêneros, optamos por criar um dicionário com a lista de gêneros no seguinte formato:

```python
{"id_item1": ["genero1", "genero2"]}
```


In [45]:
def get_genre_map(df):
    genre_map = {i['item']:i['genres'] for i in df_genres[['item', 'genres']].to_dict('records')}
    return genre_map

In [46]:
def get_user_profile_distribution(df, user, userColumn, itemColumn):
    genre_map = get_genre_map(df)
    
    # O perfil do usuário inicia como um dicionário vazio
    user_profile_distribution = {}
    # Contador para cálculo de probabilidade
    n = 0
    
    # Percorre os itens avaliados pelo usuário
    for item in df[df[userColumn] == user][itemColumn].values:
        # Busca a informação do gênero no dicionário de item: generos
        for genre in genre_map[item]:
            # se o gênero não estiver no perfil do usuário, inicia ele com 0 para iniciar a contagem nas linhas abaixo
            if genre not in user_profile_distribution:
                user_profile_distribution[genre] = 0
            # soma 1 unidade ao contador    
            n += 1
            # soma uma unidade à contagem do genero (inicialmente iniciada com 0)
            user_profile_distribution[genre] += 1

    # pega o número da contagem (indicado por v) e divide pelo contador n, para assim fazer a probabilidade. k refere-se ao gênero.
        # então temos, por exemplo -> horror: 0,25
    user_profile_distribution = {k: v/n for k, v in sorted(user_profile_distribution.items(), key=lambda item: item[1])}
    return user_profile_distribution

#### Observando distribuição graficamente

In [48]:
def show_user_profile_distribution_chart(df, user, userColumn, itemColumn):
    user_profile_distribution = get_user_profile_distribution(df, user, userColumn, itemColumn)
    
    plt.figure(figsize=(12, 8))
    sns.barplot(
        x=[i[0] for i in user_profile_distribution.items()],
        y=[i[1]*100 for i in user_profile_distribution.items()], color='gray'
    )
    plt.xticks(rotation=45)
    plt.xlabel("Generos")
    plt.ylabel("Presença do Genero no Perfil (%)")
    plt.grid()
    plt.show()